# 3. Local-GPU student experiments

This is the local Hugging Face equivalent of the OpenRouter notebook. The model is loaded once and reused across all conditions. Output files share the same resumable JSONL structure and remain inside this MMLU folder.

In [ ]:
# %pip install -r requirements.txt
from pathlib import Path
import importlib
import sys
EXPERIMENT_ROOT = Path.cwd()
if not (EXPERIMENT_ROOT / 'mmlu_common.py').exists():
    EXPERIMENT_ROOT = (Path.cwd() / 'other_experiments' / 'MMLU data').resolve()
if not (EXPERIMENT_ROOT / 'mmlu_common.py').exists():
    raise FileNotFoundError('Run this notebook from the repository root or its own folder')
sys.path.insert(0, str(EXPERIMENT_ROOT)) if str(EXPERIMENT_ROOT) not in sys.path else None
import local_gpu_experiments
importlib.reload(local_gpu_experiments)
from local_gpu_experiments import ModelConfig, run_local_experiments
print('Experiment root:', EXPERIMENT_ROOT)

## Settings
For Qwen3, `ENABLE_THINKING=False` supplies the no-thinking chat-template switch. Use a separate run with `True` if you want the reasoning comparison.

In [ ]:
MODEL = ModelConfig(
    model_id='Qwen/Qwen3-8B',
    dtype='bfloat16',
    device_map='auto',
    attention_implementation=None,  # Set 'flash_attention_2' only if installed
)
TARGET_CONDITIONS = list(range(9)) + [20]
NUM_ROWS = None
START_ROW = 0
BATCH_SIZE = 8
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.0
ENABLE_THINKING = False

## Run or resume

In [ ]:
results = run_local_experiments(
    model_config=MODEL,
    condition_ids=TARGET_CONDITIONS,
    num_rows=NUM_ROWS,
    start_row=START_ROW,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    enable_thinking=ENABLE_THINKING,
    retry_failed=True,
)
results

## Compact comparison

In [ ]:
import pandas as pd
pd.DataFrame.from_dict(results, orient='index')[
    ['requested', 'successful', 'failed', 'correct', 'accuracy', 'output_file']
].rename_axis('condition_id')